# Air Quality Prediction System — Phase 4: Feature Engineering and Data Splitting
This notebook transforms the cleaned time-series data into a supervised learning format for machine learning models. We create features from past data to predict future PM2.5 values.

## Section 1: Load Processed Data
Import necessary libraries and load the processed air quality data from 'feeds_cleaned.csv'.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib  # For saving the scaler

# --- [Phase 4] Load Processed Data ---
print("--- [Phase 4] Starting Feature Engineering ---")
try:
    df = pd.read_csv('feeds_cleaned_15T.csv', parse_dates=['created_at'], index_col='created_at')
    print("Loaded 'feeds_cleaned_15T.csv' successfully.")
    print(f"Data shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
except FileNotFoundError:
    print("ERROR: 'feeds_cleaned_15T.csv' not found. Please re-run Phase 2.")
    raise

--- [Phase 4] Starting Feature Engineering ---
Loaded 'feeds_cleaned.csv' successfully.
Data shape: (3531, 4)
Columns: ['pm2_5', 'pm10', 'temperature', 'humidity']


## Section 2: Define Target Variable
Create the target variable 'target_pm2_5' by shifting the PM2.5 values to predict the next hour.

In [ ]:
# --- [Step 1: Define Target Variable] ---
# Our goal: Predict PM2.5 15 minutes IN THE FUTURE.
# We use shift(-1) to pull the *next* 15-minute interval's PM2.5 value into the current row.
df['target_pm2_5'] = df['pm2_5'].shift(-1)
print("Target variable 'target_pm2_5' (t+15min) created.")
print(f"Target has {df['target_pm2_5'].isnull().sum()} NaN values (expected at end).")

Target variable 'target_pm2_5' (t+1) created.
Target has 1176 NaN values (expected at end).


## Section 3: Create Time-Based Features
Extract hour of day and day of week from the timestamp index to create numerical time-based features.

In [ ]:
# --- [Step 2: Create Time-Based Features] ---
# Models understand numbers, not datestamps.
df['hour_of_day'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['minute_of_hour'] = df.index.minute
print("Time-based features (hour_of_day, day_of_week, minute_of_hour) created.")
print(f"Hour range: {df['hour_of_day'].min()} to {df['hour_of_day'].max()}")
print(f"Minute range: {df['minute_of_hour'].min()} to {df['minute_of_hour'].max()}")
print(f"Day of week range: {df['day_of_week'].min()} to {df['day_of_week'].max()}")

Time-based features (hour_of_day, day_of_week) created.
Hour range: 0 to 23
Day of week range: 0 to 6


## Section 4: Create Lag Features
Generate lag features for PM2.5 and other sensors (e.g., 1h, 2h, 3h lags) using shift operations.

In [ ]:
# --- [Step 3: Create Lag Features (Past Data)] ---
# What was the PM2.5 15 minutes ago? 30 minutes ago? etc.
# These are the most important features.
df['pm2_5_lag_15m'] = df['pm2_5'].shift(1)
df['pm2_5_lag_30m'] = df['pm2_5'].shift(2)
df['pm2_5_lag_45m'] = df['pm2_5'].shift(3)
df['pm2_5_lag_60m'] = df['pm2_5'].shift(4)

# Add lags for other sensors too
df['temperature_lag_15m'] = df['temperature'].shift(1)
df['humidity_lag_15m'] = df['humidity'].shift(1)
df['pm10_lag_15m'] = df['pm10'].shift(1)
print("Lag features (t-15m, t-30m, t-45m, t-60m for PM2.5; t-15m for others) created.")
print(f"New columns added: {[c for c in df.columns if 'lag' in c]}")

Lag features (t-1, t-2, t-3 for PM2.5; t-1 for others) created.
New columns added: ['pm2_5_lag_1h', 'pm2_5_lag_2h', 'pm2_5_lag_3h', 'temperature_lag_1h', 'humidity_lag_1h', 'pm10_lag_1h']


## Section 5: Create Rolling Features
Compute rolling average features (e.g., 3-hour averages) for PM2.5 and temperature, ensuring no data leakage by shifting first.

In [ ]:
# --- [Step 4: Create Rolling Features] ---
# What was the 1-hour average PM2.5 up to this point?
# We shift(1) first to prevent data leakage (only use data *before* the current 15-min interval)
df['pm2_5_roll_avg_1h'] = df['pm2_5'].shift(1).rolling(window=4).mean()
df['temperature_roll_avg_1h'] = df['temperature'].shift(1).rolling(window=4).mean()
print("Rolling average features (1h) created for PM2.5 and temperature.")
print(f"Rolling features added: {[c for c in df.columns if 'roll' in c]}")

Rolling average features (3h) created for PM2.5 and temperature.
Rolling features added: ['pm2_5_roll_avg_3h', 'temperature_roll_avg_3h']


## Section 6: Clean Up NaNs
Drop rows with NaN values resulting from shift and rolling operations to prepare the dataset.

In [6]:
# --- [Step 5: Clean Up NaNs] ---
# All the shift() and rolling() operations created NaNs at the beginning
# and end of the dataset. We must drop them.
print(f"NaN counts before dropna: {df.isnull().sum().sum()}")
df_model = df.dropna()
print(f"Original df shape: {df.shape}, New df shape after dropna: {df_model.shape}")
print(f"Dropped {df.shape[0] - df_model.shape[0]} rows with NaNs.")

NaN counts before dropna: 15443
Original df shape: (3531, 15), New df shape after dropna: (2214, 15)
Dropped 1317 rows with NaNs.


## Section 7: Define Features and Target
Separate the features (X) and target (y), dropping original sensor columns and the target from X.

In [7]:
# --- [Step 6: Define Features (X) and Target (y)] ---
# 'y' is what we want to predict
y = df_model['target_pm2_5']

# 'X' is all the data we use to predict it.
# We drop the original sensor values and the target itself.
X = df_model.drop(['pm2_5', 'pm10', 'temperature', 'humidity', 'target_pm2_5'], axis=1)

print("\nFeatures (X) for the model:")
print(X.head())
print("\nTarget (y) for the model:")
print(y.head())
print(f"\nX shape: {X.shape}, y shape: {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")


Features (X) for the model:
                           hour_of_day  day_of_week  pm2_5_lag_1h  \
created_at                                                          
2025-03-05 06:00:00+00:00            6            2     31.596217   
2025-03-05 07:00:00+00:00            7            2     36.571015   
2025-03-05 08:00:00+00:00            8            2     36.305712   
2025-03-05 09:00:00+00:00            9            2     36.454243   
2025-03-05 10:00:00+00:00           10            2     36.926735   

                           pm2_5_lag_2h  pm2_5_lag_3h  temperature_lag_1h  \
created_at                                                                  
2025-03-05 06:00:00+00:00     31.888496     32.608883           14.535294   
2025-03-05 07:00:00+00:00     31.596217     31.888496           16.006154   
2025-03-05 08:00:00+00:00     36.571015     31.596217           16.512921   
2025-03-05 09:00:00+00:00     36.305712     36.571015           16.301705   
2025-03-05 10:00:00+00:00

## Section 8: Train-Test Split
Perform a chronological train-test split (e.g., 80% train, 20% test) without shuffling the time-series data.

In [8]:
# --- [Step 7: Train-Test Split (Chronological)] ---
# CRITICAL: We CANNOT shuffle time-series data.
# We must split it by time (e.g., first 80% for training, last 20% for testing).
split_percent = 0.8
split_point = int(len(X) * split_percent)

X_train, X_test = X.iloc[:split_point], X.iloc[split_point:]
y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]

print(f"\nData split into {split_percent*100}% train and {(1-split_percent)*100}% test.")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")
print(f"Train period: {X_train.index.min()} to {X_train.index.max()}")
print(f"Test period: {X_test.index.min()} to {X_test.index.max()}")


Data split into 80.0% train and 19.999999999999996% test.
X_train shape: (1771, 10), X_test shape: (443, 10)
y_train shape: (1771,), y_test shape: (443,)
Train period: 2025-03-05 06:00:00+00:00 to 2025-07-11 04:00:00+00:00
Test period: 2025-07-11 05:00:00+00:00 to 2025-07-30 04:00:00+00:00


## Section 9: Scale the Data
Apply StandardScaler to the training data, transform the test data, and save the scaler for future use.

In [ ]:
# --- [Step 8: Scale the Data] ---
# Models (especially Neural Networks and SVR) work best with scaled data.
# We fit the scaler ONLY on the training data to prevent data leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler so we can use it later for live predictions
joblib.dump(scaler, 'scaler_15T.pkl')
print("\nData scaled. Scaler saved to 'scaler_15T.pkl'.")
print(f"Scaler mean: {scaler.mean_}")
print(f"Scaler scale: {scaler.scale_}")


Data scaled. Scaler saved to 'scaler.pkl'.
Scaler mean: [11.5567476   2.88029362 35.12488271 35.14820106 35.15976117 18.54565763
 27.29840297 39.8634685  35.14428164 18.52555977]
Scaler scale: [ 6.83842099  1.84929401  3.03265602  3.0400932   3.05278363 11.61586742
 14.16795207  9.63653039  2.9761695  11.4185885 ]


## Section 10: Save the Final Data
Save the scaled features, targets, scaler, and feature names to files for model training.

In [11]:
# --- [Step 9: Save the Final Data] ---
# It's good practice to save the final processed data
# We'll save the scaled arrays
np.save('X_train_15T.npy', X_train_scaled)
np.save('X_test_15T.npy', X_test_scaled)
# And the target pandas Series
y_train.to_csv('y_train_15T.csv', header=True)
y_test.to_csv('y_test_15T.csv', header=True)
# Also save the feature names
joblib.dump(X.columns.to_list(), 'feature_names_15T.pkl')

print("--- [Phase 4] Feature Engineering Complete ---")
print("Data is now ready for model training.")
print("Files created: X_train_15T.npy, X_test_15T.npy, y_train_15T.csv, y_test_15T.csv, scaler_15T.pkl, feature_names_15T.pkl")
print(f"\nFinal dataset summary:")
print(f"- Training samples: {len(X_train)}")
print(f"- Test samples: {len(X_test)}")
print(f"- Features: {len(X.columns)}")
print(f"- Target range: {y.min():.2f} to {y.max():.2f}")

--- [Phase 4] Feature Engineering Complete ---
Data is now ready for model training.
Files created: X_train_15T.npy, X_test_15T.npy, y_train_15T.csv, y_test_15T.csv, scaler_15T.pkl, feature_names_15T.pkl

Final dataset summary:
- Training samples: 1771
- Test samples: 443
- Features: 10
- Target range: 21.76 to 43.68
